# Perturbation study

Combined results for **F1–F5**, using all complete **5 ms windows in the first rotation** and all events per window. Rows are perturbations; columns are metrics. Colours match the earlier Test 3 figure.

Spatial offset, scaling and jitter use the completed expanded sweeps from 22 September. Subsampling, uniform noise and temporal offset retain the completed 21 September results. The expanded temporal run was stopped at the user's request; its partial results are excluded. This notebook reads saved results only—it does not rerun evaluations.


In [1]:
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import yaml
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

pio.renderers.default = "plotly_mimetype"

# Works when the notebook is launched from the repository or its notebook folder.
REPO = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "experiments" / "perturbation.py").is_file()
)
RESULTS_ROOT = REPO / "output" / "perturbation"
UPDATED_RUN = RESULTS_ROOT / "20260922_181016_100519"
SUITE_RUN = RESULTS_ROOT / "20260921_210141_079777"
NOTEBOOK_OUTPUT = RESULTS_ROOT / "notebook_20260923_spatial_refresh"
NOTEBOOK_OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURE_PATH = NOTEBOOK_OUTPUT / "perturbation_overview.html"

FREQUENCIES = ["F1", "F2", "F3", "F4", "F5"]
FREQUENCY_COLOURS = {
    "F1": "#d1352b", "F2": "#e07b28", "F3": "#2f9e44",
    "F4": "#1f6fb4", "F5": "#7b3ea1",
}
METRIC_ORDER = ["mmd_rbf03", "mmd_rbf15", "mmd_rbf75", "swd", "chamfer"]
METRIC_LABELS = {
    "mmd_rbf03": "MMD RBF-3", "mmd_rbf15": "MMD RBF-15",
    "mmd_rbf75": "MMD RBF-75", "swd": "SWD", "chamfer": "Chamfer",
}
MODIFIER_ORDER = [
    "spatial_offset", "spatial_scaling", "spatial_jitter",
    "subsampling", "uniform_noise", "temporal_offset",
]
MODIFIER_LABELS = {
    "spatial_offset": "Spatial offset x+y<br>(pixels per axis)",
    "spatial_scaling": "Spatial scaling<br>(factor)",
    "spatial_jitter": "Spatial jitter<br>(max pixels per axis)",
    "subsampling": "Subsampling<br>(retained fraction)",
    "uniform_noise": "Uniform noise<br>(added/original count)",
    "temporal_offset": "Temporal phase offset (previous sweep)<br>(rotation degrees)",
}
MAGNITUDE_LABELS = {
    "spatial_offset": "Offset per axis (px)",
    "spatial_scaling": "Scale factor",
    "spatial_jitter": "Maximum jitter per axis (px)",
    "subsampling": "Retained fraction",
    "uniform_noise": "Added/original event ratio",
    "temporal_offset": "Rotation offset (degrees)",
}

# Whiskers describe variation across windows, not confidence intervals.
SHOW_WINDOW_SPREAD = True


## Load completed method results

Each trial/method is validated for complete comparison coverage, unique keys, statuses, identities, event counts, signed MMD values and recomputed summaries before plotting. Measurement settings and rotation periods must agree across sources. Only the three completed spatial methods are loaded from the stopped run; temporal results remain from the previous completed sweep. Signed MMD-squared values and exact source paths remain available in `perturbation_summary`.


In [2]:
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from analysis.consolidate_perturbation import read_yaml, settings_signature, validate_trial

METHOD_SOURCES = {
    method: UPDATED_RUN if method in ("spatial_offset", "spatial_scaling", "spatial_jitter") else SUITE_RUN
    for method in MODIFIER_ORDER
}


def load_perturbation_summary() -> pd.DataFrame:
    """Load only complete method outputs; never include the interrupted temporal sweep."""
    config = read_yaml(UPDATED_RUN / "run_config.yaml")
    signature = settings_signature(config)
    if settings_signature(read_yaml(SUITE_RUN / "run_config.yaml")) != signature:
        raise ValueError("Source runs use incompatible measurement settings.")
    if config["arguments"]["temporal_window"] != 5000:
        raise ValueError("Update the figure caption for this window length.")
    trials = config["arguments"]["trials"]
    summaries = []
    for trial, method in product(trials, MODIFIER_ORDER):
        block, audit = validate_trial(METHOD_SOURCES[method], trial, method, signature)
        summaries.append(block)
    summary = pd.concat(summaries, ignore_index=True)
    if summary.groupby("trial").rotation_period_us.nunique().ne(1).any():
        raise ValueError("Rotation periods disagree between source runs.")
    summary["frequency"] = summary.trial.str.extract(r"_f([1-5])$", expand=False).map(
        lambda number: f"F{number}"
    )
    keys = ["perturbation", "metric", "frequency", "magnitude"]
    if summary.duplicated(keys).any():
        raise ValueError("Duplicate summary conditions found.")
    expected = set(product(MODIFIER_ORDER, METRIC_ORDER, FREQUENCIES))
    actual = set(summary[["perturbation", "metric", "frequency"]].itertuples(index=False, name=None))
    if actual != expected:
        raise ValueError("Missing or unexpected perturbation/metric/recording groups.")
    if not np.isfinite(summary.mean_distance).all():
        raise ValueError("Non-finite summary means found; inspect exclusions.")
    return summary


perturbation_summary = load_perturbation_summary()
print(
    f"Loaded {len(perturbation_summary):,} validated summary rows: "
    f"{len(MODIFIER_ORDER)} perturbations, {len(METRIC_ORDER)} metrics, "
    f"{len(FREQUENCIES)} recordings."
)
print(f"Excluded comparisons: {perturbation_summary.n_excluded.sum():,}")
for method, source in METHOD_SOURCES.items():
    print(f"{method}: {source.name}")
print("Temporal offset uses the previous completed sweep; expanded partial results are excluded.")


Loaded 1,725 validated summary rows: 6 perturbations, 5 metrics, 5 recordings.
Excluded comparisons: 0
spatial_offset: 20260922_181016_100519
spatial_scaling: 20260922_181016_100519
spatial_jitter: 20260922_181016_100519
subsampling: 20260921_210141_079777
uniform_noise: 20260921_210141_079777
temporal_offset: 20260921_210141_079777
Temporal offset uses the previous completed sweep; expanded partial results are excluded.


## Combined perturbation response

Lines show the mean distance across windows; thin whiskers show the **5th–95th window percentiles**, not confidence intervals or significance thresholds. Set `SHOW_WINDOW_SPREAD = False` for a cleaner view. Each panel keeps its own vertical scale in the metric's raw units.

- Offset adds the stated value to **both x and y**; jitter is independent uniform displacement within ±magnitude on each axis.
- Noise magnitude is added/original count: 0.5 gives a final noise fraction of one third.
- Temporal offset compares later recorded windows, re-zeroed to their own start; 360° compares the next rotation and need not give zero.
- MMD distances use `sqrt(max(MMD², 0))`; a flat zero curve can hide changes in signed estimates. No old null-test reference lines are reused.


In [3]:
def add_recording_curve(figure, block, row, col, modifier, frequency, show_spread):
    """Add one recording's mean curve and optional descriptive window whiskers."""
    block = block.sort_values("magnitude")
    colour = FREQUENCY_COLOURS[frequency]
    if show_spread:
        # Separate segments preserve the actual quantiles even when the mean is
        # outside their interval (possible for strongly skewed/clamped results).
        whisker_x = np.column_stack([
            block.magnitude, block.magnitude, np.full(len(block), np.nan)
        ]).ravel()
        whisker_y = np.column_stack([
            block.q05_distance, block.q95_distance, np.full(len(block), np.nan)
        ]).ravel()
        figure.add_trace(go.Scatter(
            x=whisker_x, y=whisker_y, mode="lines",
            line=dict(color=colour, width=0.6), opacity=0.45,
            legendgroup=frequency, showlegend=False, hoverinfo="skip",
        ), row=row, col=col)
    figure.add_trace(go.Scatter(
        x=block.magnitude, y=block.mean_distance, mode="lines+markers",
        name=frequency, legendgroup=frequency,
        showlegend=(row == 1 and col == 1),
        line=dict(color=colour, width=1.8), marker=dict(size=4),
        customdata=block[[
            "q05_distance", "q95_distance", "n_valid",
            "n_excluded", "mean_distance_squared", "source_run",
        ]].to_numpy(),
        hovertemplate=(
            frequency + "<br>" + MAGNITUDE_LABELS[modifier] + ": %{x:g}"
            "<br>Mean distance: %{y:.5g}"
            "<br>Window 5–95%: %{customdata[0]:.5g} – %{customdata[1]:.5g}"
            "<br>Valid: %{customdata[2]:,.0f}; excluded: %{customdata[3]:,.0f}"
            "<br>Mean signed MMD² (MMD only): %{customdata[4]:.5g}"
            "<br>Run: %{customdata[5]}<extra></extra>"
        ),
    ), row=row, col=col)


def add_metric_panel(figure, summary, modifier, metric, row, col, show_spread):
    """Populate one perturbation/metric panel with F1–F5 in the original colours."""
    panel = summary[
        (summary.perturbation == modifier) & (summary.metric == metric)
    ]
    for frequency in FREQUENCIES:
        block = panel[panel.frequency == frequency]
        add_recording_curve(figure, block, row, col, modifier, frequency, show_spread)
    figure.update_yaxes(rangemode="tozero", row=row, col=col)
    if col == 1:
        figure.update_yaxes(title_text=MODIFIER_LABELS[modifier], row=row, col=col)
    if modifier == "temporal_offset":
        figure.update_xaxes(tickvals=[0, 90, 180, 270, 360], row=row, col=col)


def perturbation_overview(summary, show_spread=True):
    """Build a six-row, five-column overview without recomputing any metrics."""
    figure = make_subplots(
        rows=len(MODIFIER_ORDER), cols=len(METRIC_ORDER),
        horizontal_spacing=0.035, vertical_spacing=0.055,
        column_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
    )
    panels = product(enumerate(MODIFIER_ORDER, 1), enumerate(METRIC_ORDER, 1))
    for (row, modifier), (col, metric) in panels:
        add_metric_panel(figure, summary, modifier, metric, row, col, show_spread)
    spread_note = "; whiskers: window 5–95%" if show_spread else ""
    figure.update_layout(
        title=(
            "Perturbation responses across F1–F5"
            "<br><sup>5 ms windows; mean distance in raw metric units; colour identifies recording"
            + spread_note + "</sup>"
        ),
        template="plotly_white", height=225 * len(MODIFIER_ORDER), width=1250,
        margin=dict(l=155, r=145, t=105, b=50),
        legend=dict(orientation="v", xanchor="left", x=1.01,
                    yanchor="top", y=1.0, title_text="Recording"),
    )
    return figure


overview_figure = perturbation_overview(perturbation_summary, SHOW_WINDOW_SPREAD)
overview_figure.write_html(FIGURE_PATH, include_plotlyjs=True)
display(overview_figure)
print(f"Saved interactive figure: {FIGURE_PATH}")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\perturbation_overview.html


## Individual perturbations

The following figures reuse the same saved means, recording colours and optional percentile whiskers. Each non-temporal perturbation has one row of five metric panels. Temporal offset uses one wide panel per metric, stacked vertically.


In [4]:
SEPARATE_FIGURE_DIR = NOTEBOOK_OUTPUT / "notebook_figures"


def single_perturbation_figure(summary, modifier, show_spread=True):
    """Plot one non-temporal perturbation with a column for each metric."""
    figure = make_subplots(
        rows=1, cols=len(METRIC_ORDER), horizontal_spacing=0.045,
        column_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
    )
    x_label = MAGNITUDE_LABELS[modifier]
    if modifier == "spatial_jitter":
        x_label = "Maximum jitter<br>per axis (px)"
    for col, metric in enumerate(METRIC_ORDER, 1):
        add_metric_panel(figure, summary, modifier, metric, 1, col, show_spread)
        figure.update_xaxes(title_text=x_label, row=1, col=col)
        figure.update_yaxes(title_text="Mean distance" if col == 1 else None, row=1, col=col)
    spread_note = "; whiskers: window 5–95%" if show_spread else ""
    title = MODIFIER_LABELS[modifier].split("<br>")[0]
    figure.update_layout(
        title=(
            f"{title}: responses across F1–F5"
            "<br><sup>5 ms windows; mean distance in raw metric units"
            + spread_note + "</sup>"
        ),
        template="plotly_white", width=1450, height=420,
        margin=dict(l=80, r=140, t=105, b=85),
        legend=dict(orientation="v", xanchor="left", x=1.01,
                    yanchor="top", y=1.0, title_text="Recording"),
    )
    return figure


def temporal_offset_figure(summary, show_spread=True):
    """Plot phase response with one wide row per metric and a shared degree axis."""
    figure = make_subplots(
        rows=len(METRIC_ORDER), cols=1, shared_xaxes=True,
        vertical_spacing=0.045,
        subplot_titles=[METRIC_LABELS[metric] for metric in METRIC_ORDER],
    )
    for row, metric in enumerate(METRIC_ORDER, 1):
        add_metric_panel(
            figure, summary, "temporal_offset", metric, row, 1, show_spread
        )
        figure.update_yaxes(title_text="Mean distance", row=row, col=1)
        figure.update_xaxes(
            tickvals=list(range(0, 361, 45)), range=[0, 360], row=row, col=1
        )
    figure.update_xaxes(
        title_text=MAGNITUDE_LABELS["temporal_offset"],
        row=len(METRIC_ORDER), col=1,
    )
    spread_note = "; whiskers: window 5–95%" if show_spread else ""
    figure.update_layout(
        title=(
            "Temporal phase offset (previous sweep): responses across F1–F5"
            "<br><sup>5 ms windows; mean distance in raw metric units"
            + spread_note + "</sup>"
        ),
        template="plotly_white", width=1450, height=1150,
        margin=dict(l=85, r=140, t=105, b=65),
        legend=dict(orientation="v", xanchor="left", x=1.01,
                    yanchor="top", y=1.0, title_text="Recording"),
    )
    return figure


def show_and_save_perturbation(figure, modifier):
    """Display a notebook figure and save HTML separately from the run's original plots."""
    SEPARATE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = SEPARATE_FIGURE_DIR / f"{modifier}.html"
    figure.write_html(path, include_plotlyjs=True)
    display(figure)
    print(f"Saved interactive figure: {path}")


### Spatial offset


In [5]:
spatial_offset_plot = single_perturbation_figure(
    perturbation_summary, "spatial_offset", show_spread=SHOW_WINDOW_SPREAD
)
show_and_save_perturbation(spatial_offset_plot, "spatial_offset")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\notebook_figures\spatial_offset.html


### Spatial scaling


In [6]:
spatial_scaling_plot = single_perturbation_figure(
    perturbation_summary, "spatial_scaling", show_spread=SHOW_WINDOW_SPREAD
)
show_and_save_perturbation(spatial_scaling_plot, "spatial_scaling")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\notebook_figures\spatial_scaling.html


### Spatial jitter


In [7]:
spatial_jitter_plot = single_perturbation_figure(
    perturbation_summary, "spatial_jitter", show_spread=SHOW_WINDOW_SPREAD
)
show_and_save_perturbation(spatial_jitter_plot, "spatial_jitter")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\notebook_figures\spatial_jitter.html


### Subsampling


In [8]:
subsampling_plot = single_perturbation_figure(
    perturbation_summary, "subsampling", show_spread=SHOW_WINDOW_SPREAD
)
show_and_save_perturbation(subsampling_plot, "subsampling")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\notebook_figures\subsampling.html


### Uniform noise


In [9]:
uniform_noise_plot = single_perturbation_figure(
    perturbation_summary, "uniform_noise", show_spread=SHOW_WINDOW_SPREAD
)
show_and_save_perturbation(uniform_noise_plot, "uniform_noise")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\notebook_figures\uniform_noise.html


### Temporal offset — previous completed sweep

The expanded temporal evaluation was stopped on 23 September. The figure below retains the earlier complete F1–F5 sweep, not the incomplete expanded results. Temporal sampling strategy will be reconsidered separately.


In [10]:
temporal_offset_plot = temporal_offset_figure(
    perturbation_summary, show_spread=SHOW_WINDOW_SPREAD
)
show_and_save_perturbation(temporal_offset_plot, "temporal_offset")


Saved interactive figure: C:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis\output\perturbation\notebook_20260923_spatial_refresh\notebook_figures\temporal_offset.html
